In [ ]:
# Cell 1: Environment Setup & Package Imports
import sys
from pathlib import Path
import json
import numpy as np
import pandas as pd

REPO_ROOT = Path("..").resolve() if Path("..").joinpath("src").exists() else Path(".").resolve()
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

import fragility
from fragility import (
    SOGConfig,
    evaluate_gold20,
    evaluate_ablation,
    evaluate_grid_stability,
    evaluate_state_fragility,
    exact_permutation_auc,
    plot_gold20_recovery,
    plot_grid_stability,
)
print(f"Loaded fragility v{fragility.__version__} from {REPO_ROOT}")


In [ ]:
# Cell 2: Mode Selection & Canonical Baseline Loading
# Mode: 'public' reproduces all claims from shareable derived tables.
#       'full' executes end-to-end if PFF_DATA_ROOT is provided.
MODE = "public"
DATA_DIR = REPO_ROOT / "data"
RESULTS_DIR = REPO_ROOT / "results"
with open(RESULTS_DIR / "expected_results.json", encoding="utf-8") as f:
    EXPECTED = json.load(f)
print(f"Mode: {MODE.upper()} | Baseline: {EXPECTED['metadata']['paper_title']}")


In [ ]:
# Cell 3: Canonical SOG Configuration (SSAC27 / db10578)
cfg = SOGConfig()
print("SOG Canonical Parameters (db10578):")
print(f"  Pitch: {cfg.pitch_length}m x {cfg.pitch_width}m | Grid: {cfg.grid_nx}x{cfg.grid_ny} cells")
print(f"  Perturbation: {len(cfg.radii)} radii {cfg.radii}m x {len(cfg.directions)} directions = 24 actions/player")
print(f"  Kinematics: v_max={cfg.max_player_speed}m/s, a_max={cfg.max_acceleration}m/s2, carrier_dist={cfg.carrier_max_distance_m}m")
print(f"  Offside Tol: {cfg.offside_tol_m}m (offside-switch exclusion) | Collision: {cfg.collision_radius_m}m")
print(f"  Fragility Reducer: arithmetic mean of top {int(cfg.tail_fraction*100)}% valid actions")


In [ ]:
# Cell 4: Tactical Ground-Truth Recovery (Claims C01 & C02 - EXP023)
pa_df = pd.read_csv(DATA_DIR / "derived" / "gold20_player_actions.csv")
cs_df = pd.read_csv(DATA_DIR / "derived" / "gold20_cases_summary.csv")
c01_c02 = evaluate_gold20(pa_df, cs_df)
bank_a = c01_c02["A"]
bank_b = c01_c02["B"]

print(f"[C01] Ground-Truth Action Recovery (Bank A): {bank_a['top10_supported']}/{bank_a['evaluable_actions']} "
      f"({bank_a['action_rate']*100:.1f}%) 95% Wilson CI: [{bank_a['wilson95'][0]*100:.1f}%, {bank_a['wilson95'][1]*100:.1f}%]")
print(f"      Median Peak Percentile: {bank_a['median_peak']*100:.1f}% | Median Window: {bank_a['median_window']*100:.1f}%")
print(f"[C02] Space-Creator / Decoy Recovery: {bank_a['space_creator_hits']}/{bank_a['space_creator_n']} ({bank_a['space_creator_rate']*100:.1f}%)")
print(f"      Primary Exploiter Recovery: {bank_a['primary_exploiter_hits']}/{bank_a['primary_exploiter_n']} ({bank_a['primary_exploiter_rate']*100:.1f}%)")
print(f"      Secondary Bank B Recovery: {bank_b['top10_supported']}/{bank_b['evaluable_actions']} ({bank_b['action_rate']*100:.1f}%)")


In [ ]:
# Cell 5: Incremental Residual Value Ablation (Claim C03 - EXP024)
abl_df = pd.read_csv(DATA_DIR / "derived" / "gold20_ablation_comparison.csv")
c03 = evaluate_ablation(abl_df)
print(f"[C03] SOG Recovery Rate: {c03['sog_hits']}/{c03['n_evaluable']} ({c03['sog_rate']*100:.1f}%)")
print(f"      PCG (Pitch Control Gain) Rate: {c03['pcg_hits']}/{c03['n_evaluable']} ({c03['pcg_rate']*100:.1f}%)")
print(f"      Incremental Value of (1 - C0) Residual Weight: +{c03['delta_pp']*100:.1f} pp")
print(f"      1,000-resample Case Bootstrap 95% CI: [{c03['bootstrap_95_ci'][0]*100:.1f}%, {c03['bootstrap_95_ci'][1]*100:.1f}%]")
print(f"      Median Peak: SOG={c03['median_peak_sog']*100:.1f}% vs PCG={c03['median_peak_pcg']*100:.1f}%")


In [ ]:
# Cell 6: Discretization Invariance / Grid Convergence (Claim C04 - EXP026A)
stab_df = pd.read_csv(DATA_DIR / "derived" / "grid_stability_summary.csv")
c04 = evaluate_grid_stability(stab_df)
print(f"[C04] G0 (50x32, 1,600 cells) vs G1 (100x64, 6,400 cells) Convergence across {c04['n_states']} states:")
print(f"      Median Action Spearman rho: {c04['median_spearman']:.4f} (Min: {c04['min_spearman']:.4f})")
print(f"      Median Top-10% Jaccard:     {c04['median_top10_jaccard']:.4f} (Min: {c04['min_top10_jaccard']:.4f})")
print(f"      Best-Player Agreement:      {c04['best_player_agreement']*100:.1f}%")
print(f"      Top-1 Action Exact Match:   {c04['top1_agreement']*100:.1f}%")


In [ ]:
# Cell 7: State Fragility Diagnostic on CAP Matched Pairs (Claim C05 - EXP027)
cap_df = pd.read_csv(DATA_DIR / "derived" / "cap_pair_scores.csv")
c05 = evaluate_state_fragility(cap_df)
print(f"[C05] State-Level Fragility & Path Accessibility on {c05['n_pairs']} Matched Pairs:")
print(f"      Win-Share F_SOG:    {c05['win_share_F_SOG']*100:.2f}% (14/32 pairs)")
print(f"      Win-Share F_PA:     {c05['win_share_F_PA']*100:.2f}% (17/32 pairs)")
print(f"      Win-Share F_RADIAL: {c05['win_share_F_RADIAL']*100:.2f}% (17/32 pairs)")
print(f"      Delta (PA - RADIAL): {c05['delta_PA_RADIAL']:.4f} ({c05['agree_pa_rad_count']}/32 pairs identical)")
print(f"      Scientific Verdict: {c05['verdict']} (Angular path accessibility adds zero incremental signal)")


In [ ]:
# Cell 8: Independent Fragility Validation Preregistration (Claim C06 - EXP028)
frag_val = pd.read_csv(DATA_DIR / "validation" / "fragility_validation.csv")
exp028 = EXPECTED["claims"]["C06_fragility_construct_validation"]
n_v = int((frag_val["label"] == "VULNERABLE").sum())
n_r = int((frag_val["label"] == "ROBUST").sum())
print(f"[C06] Independent Construct Validation (Status: {exp028['status']}):")
print(f"      Validation Cohort: {len(frag_val)} states ({n_v} vulnerable, {n_r} robust across 2022 World Cup)")
print(f"      Preregistration Protocol: {exp028['test']}")
print(f"      Full Solve: 203 states ({exp028['canonical_experiment']})")


In [ ]:
# Cell 9: Publication Figures Regeneration
fig_dir = RESULTS_DIR / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)
p1 = plot_gold20_recovery(abl_df, fig_dir / "figure1_gold20_recovery.png")
p2 = plot_grid_stability(stab_df, fig_dir / "figure2_grid_stability.png")
print("Regenerated Publication Figures:")
print(f"  Figure 1: {p1} ({p1.stat().st_size / 1024:.1f} KB)")
print(f"  Figure 2: {p2} ({p2.stat().st_size / 1024:.1f} KB)")


In [ ]:
# Cell 10: Automated Self-Verification Matrix against Canonical Results
checks = [
    ("C01 Gold-20 Action Recovery Rate", np.isclose(bank_a["action_rate"], EXPECTED["claims"]["C01_gold20_action_recovery"]["a_bank"]["action_rate"], atol=1e-5), f"{bank_a['action_rate']*100:.1f}% == 65.0%"),
    ("C02 Space-Creator Recovery Rate", np.isclose(bank_a["space_creator_rate"], EXPECTED["claims"]["C02_gold20_space_creators"]["space_creator_rate"], atol=1e-5), f"{bank_a['space_creator_rate']*100:.1f}% == 100.0%"),
    ("C03 Residual Ablation Delta", np.isclose(c03["delta_pp"], EXPECTED["claims"]["C03_residual_weight_ablation"]["delta_rate_pp"], atol=1e-5), f"+{c03['delta_pp']*100:.1f} pp == +10.0 pp"),
    ("C04 Grid Median Spearman Rho", np.isclose(c04["median_spearman"], EXPECTED["claims"]["C04_grid_convergence"]["median_action_spearman"], atol=1e-4), f"{c04['median_spearman']:.4f} == 0.9998"),
    ("C04 Grid Best-Player Agreement", np.isclose(c04["best_player_agreement"], EXPECTED["claims"]["C04_grid_convergence"]["best_player_agreement"], atol=1e-5), f"{c04['best_player_agreement']*100:.1f}% == 100.0%"),
    ("C05 CAP Matched-Pairs Delta PA-RAD", np.isclose(c05["delta_PA_RADIAL"], EXPECTED["claims"]["C05_state_fragility_path_accessibility"]["delta_PA_RADIAL"], atol=1e-6), f"{c05['delta_PA_RADIAL']:.4f} == 0.0000"),
    ("C05 CAP Distance-Only Verdict", c05["verdict"] == EXPECTED["claims"]["C05_state_fragility_path_accessibility"]["verdict"], f"{c05['verdict']} == DISTANCE_EFFECT_ONLY"),
    ("C06 Validation Cohort Setup", len(frag_val) == 11 and n_v == 9 and n_r == 2, f"{len(frag_val)} states ({n_v} vuln, {n_r} robust)"),
]

print("=" * 76)
print("             SSAC27 CANONICAL CLAIM REPRODUCTION MATRIX             ")
print("=" * 76)
all_pass = True
for name, passed, detail in checks:
    status = "[PASS]" if passed else "[FAIL]"
    if not passed:
        all_pass = False
    print(f"{status:<8} {name:<35} : {detail}")
print("=" * 76)
assert all_pass, "One or more claim verifications failed!"
print("[OK] SSAC27 reproduction passed successfully across all published claims.")
